In [31]:
import pandas as pd
from datasets import load_dataset
import json
from pathlib import Path

from sklearn.model_selection import train_test_split

D:\PythonProjects\Fine-tuning-LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

In [33]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
        num_rows: 61765
    })
})


In [34]:
dataset['train'][0]

{'subject': 'Wesentlicher Sicherheitsvorfall',
 'body': 'Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.',
 'answer': 'Vielen Dank für die Meldung des kritischen Sicherheitsvorfalls und die Bereitstellung der Übersicht über die betroffenen Geräte sowie der ergriffenen ersten Maßnahmen. Wir e

In [35]:
dataset_list = []

for i in range(dataset['train'].num_rows):
    dataset_row = dataset['train'][i]
    dataset_list.append(dataset_row)

In [36]:
dataframe = pd.DataFrame(dataset_list)

In [37]:
dataframe.isna().sum()

subject      5299
body            2
answer      13189
type        13178
queue           0
priority        0
language        0
version     33178
tag_1       13178
tag_2       13237
tag_3       13409
tag_4       17775
tag_5       34129
tag_6       48540
tag_7       55797
tag_8       59293
dtype: int64

In [38]:
tuple(dataframe['language'].unique())

('de', 'en')

In [39]:
print(f"Size of dataframe: {len(dataframe)}")
print(f"Num columns: {len(dataframe.columns)}")
print(f"Columns name: {dataframe.columns}")
print(f"Num languages: {dataframe['language'].nunique()}")
print(f"Languages: {', '.join(dataframe['language'].unique())}")

dataframe.head()

Size of dataframe: 61765
Num columns: 16
Columns name: Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6',
       'tag_7', 'tag_8'],
      dtype='str')
Num languages: 2
Languages: de, en


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [40]:
dataframe['language'].value_counts()

language
de    33504
en    28261
Name: count, dtype: int64

In [41]:
df_en = dataframe[dataframe['language'] == 'en']
df_en.shape

(28261, 16)

In [42]:
df_en.iloc[0]

subject                                    Account Disruption
body        Dear Customer Support Team,\n\nI am writing to...
answer      Thank you for reaching out, <name>. We are awa...
type                                                 Incident
queue                                       Technical Support
priority                                                 high
language                                                   en
version                                                  51.0
tag_1                                                 Account
tag_2                                              Disruption
tag_3                                                  Outage
tag_4                                                      IT
tag_5                                            Tech Support
tag_6                                                     NaN
tag_7                                                     NaN
tag_8                                                     NaN
Name: 1,

In [43]:
def build_input(row: pd.Series) -> str:
    for col in ['subject', 'body']:
        if col not in row.index:
            raise ValueError(f"Отсутствует колонка {col}")

    subject = str(row.get('subject', "")).strip()
    body = str(row.get('body', "")).replace("\\n", "\n").strip()
    return f"Subject: {subject}\n\nBody: {body}"

In [44]:
build_input(df_en.iloc[0])

'Subject: Account Disruption\n\nBody: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?'

In [45]:
print(build_input(df_en.iloc[0]))

Subject: Account Disruption

Body: Dear Customer Support Team,

I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.

Could you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?


In [46]:
df_en.iloc[0]

subject                                    Account Disruption
body        Dear Customer Support Team,\n\nI am writing to...
answer      Thank you for reaching out, <name>. We are awa...
type                                                 Incident
queue                                       Technical Support
priority                                                 high
language                                                   en
version                                                  51.0
tag_1                                                 Account
tag_2                                              Disruption
tag_3                                                  Outage
tag_4                                                      IT
tag_5                                            Tech Support
tag_6                                                     NaN
tag_7                                                     NaN
tag_8                                                     NaN
Name: 1,

In [47]:
def build_tags(row: pd.Series) -> list[str]:
    tag_cols = [f"tag_{i}" for i in range(1, 9)]

    tags = []

    for col in tag_cols:
        if col not in row.index:
            continue

        value = row.get(col)

        if pd.isna(value):
            continue

        value = str(value).strip()

        if not value:
            continue

        tags.append(value)

    return tags

In [48]:
def build_target(row: pd.Series) -> dict:
    for col in ['type', 'queue', 'priority']:
        if col not in row.index:
            raise ValueError(f"Отсутствует колонка {col}")

    return {
        "ticket_type": str(row.get('type', "")).strip(),
        "topic": str(row.get('queue', "")).strip(),
        "urgency": str(row.get('priority', "")).strip(),
        "tags": build_tags(row)
    }

In [49]:
build_target(df_en.iloc[0])

{'ticket_type': 'Incident',
 'topic': 'Technical Support',
 'urgency': 'high',
 'tags': ['Account', 'Disruption', 'Outage', 'IT', 'Tech Support']}

In [21]:
SYSTEM_PROMPT = (
    "You extract structured routing information from customer support tickets. "
    "Return only valid JSON with fields: ticket_type, topic, urgency, tags."
)

In [22]:
def build_chat_example(row: pd.Series) -> dict:
    input_text = build_input(row)
    target = build_target(row)

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": input_text,
            },
            {
                "role": "assistant",
                "content": json.dumps(target, ensure_ascii=False)
            }
        ],
        "target": target,
    }

In [23]:
example = build_chat_example(df_en.iloc[0])
example

{'messages': [{'role': 'system',
   'content': 'You extract structured routing information from customer support tickets. Return only valid JSON with fields: ticket_type, topic, urgency, tags.'},
  {'role': 'user',
   'content': 'Subject: Account Disruption\n\nBody: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?'},
  {'role': 'assistant',
   'content': '{"ticket_type": "Incident", "topic": "Technical Support", "urgency": "high", "tags": ["Account", "Disruption", "Outage", "IT", "Tech Support"]}'}]

In [24]:
examples = [build_chat_example(row) for _, row in df_en.iterrows()]

In [25]:
print(len(examples))
print(examples[0])

28261
{'messages': [{'role': 'system', 'content': 'You extract structured routing information from customer support tickets. Return only valid JSON with fields: ticket_type, topic, urgency, tags.'}, {'role': 'user', 'content': 'Subject: Account Disruption\n\nBody: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?'}, {'role': 'assistant', 'content': '{"ticket_type": "Incident", "topic": "Technical Support", "urgency": "high", "tags": ["Account", "Disruption", "Outage", "IT", "Tech Support"]}'}], 'targ

In [26]:
train_rows, temp_rows = train_test_split(
    examples,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

validation_rows, test_rows = train_test_split(
    temp_rows,
    test_size=0.5,
    random_state=42,
    shuffle=True,
)

print(f"train: {len(train_rows)}")
print(f"validation: {len(validation_rows)}")
print(f"test: {len(test_rows)}")

train: 22608
validation: 2826
test: 2827


In [104]:
def save_jsonl(rows: list[dict], path: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [81]:
save_jsonl(train_rows, "data/processed/train.jsonl")
save_jsonl(validation_rows, "data/processed/validation.jsonl")
save_jsonl(test_rows, "data/processed/test.jsonl")

In [27]:
from src.schemas import TicketRoutingOutput

In [30]:
raw = {
    "ticket_type": "Request",
    "topic": "Technical Support",
    "urgency": "high",
    "tags": ["Product", "Feature"]
}

obj = TicketRoutingOutput.model_validate(raw)

print(obj)
print(obj.model_dump())

ticket_type='Request' topic='Technical Support' urgency='high' tags=['Product', 'Feature']
{'ticket_type': 'Request', 'topic': 'Technical Support', 'urgency': 'high', 'tags': ['Product', 'Feature']}


In [31]:
bad_raw = {
    "ticket_type": "Something",
    "topic": "Technical Support",
    "urgency": "urgent",
    "tags": []
}

TicketRoutingOutput.model_validate(bad_raw)

ValidationError: 2 validation errors for TicketRoutingOutput
ticket_type
  Input should be 'Incident', 'Request', 'Problem' or 'Change' [type=literal_error, input_value='Something', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
urgency
  Input should be 'low', 'medium', 'high' or 'critical' [type=literal_error, input_value='urgent', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error

In [33]:
PATH_TRAIN = r"D:\PythonProjects\Fine-tuning-LLM\data\processed\train.jsonl"
PATH_VALIDATION = r"D:\PythonProjects\Fine-tuning-LLM\data\processed\validation.jsonl"
PATH_TEST = r"D:\PythonProjects\Fine-tuning-LLM\data\processed\test.jsonl"

paths = [
    Path(PATH_TRAIN),
    Path(PATH_VALIDATION),
    Path(PATH_TEST)
]


In [35]:
for path in paths:
    total = 0
    errors = 0

    with path.open('r', encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            target = row['target']

            try:
                TicketRoutingOutput.model_validate(target)
            except Exception as e:
                errors += 1

            assistant_content = row['messages'][2]['content']
            assistant_json = json.loads(assistant_content)

            try:
                TicketRoutingOutput.model_validate(assistant_json)
            except Exception as e:
                errors += 1

            total += 1

    print(f"{path}: {total=}, {errors=}")

D:\PythonProjects\Fine-tuning-LLM\data\processed\train.jsonl: total=1200, errors=0
D:\PythonProjects\Fine-tuning-LLM\data\processed\validation.jsonl: total=150, errors=0
D:\PythonProjects\Fine-tuning-LLM\data\processed\test.jsonl: total=150, errors=0
